In [1]:
from pathlib import Path
import json
import shutil
import pandas as pd
import numpy as np
import xgboost as xgb

PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT.name != "Fraud-detection-ML-V2" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_CANDIDATE_PATH = (
    MODELS_DIR
    / "xgboost_final_candidate.json"
)

FINAL_EVALUATION_PATH = (
    METRICS_DIR
    / "final_model_evaluation.json"
)

TEST_FEATURE_PATH = (
    PROCESSED_DATA_DIR
    / "ml_test_features.csv"
)

if not FINAL_MODEL_CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        f"Final model candidate not found:\n{FINAL_MODEL_CANDIDATE_PATH}"
    )

if not FINAL_EVALUATION_PATH.exists():
    raise FileNotFoundError(
        f"Final evaluation file not found:\n{FINAL_EVALUATION_PATH}"
    )

if not TEST_FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Test feature file not found:\n{TEST_FEATURE_PATH}"
    )

with open(FINAL_EVALUATION_PATH, "r") as file:
    final_evaluation = json.load(file)

df = pd.read_csv(
    TEST_FEATURE_PATH
)

ML_FEATURES = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

print("========== PHASE 8 INPUTS ==========")
print(
    "Final candidate exists:",
    FINAL_MODEL_CANDIDATE_PATH.exists()
)
print(
    "Evaluation file exists:",
    FINAL_EVALUATION_PATH.exists()
)
print(
    "Test feature file exists:",
    TEST_FEATURE_PATH.exists()
)
print(
    "Selected model:",
    final_evaluation["selected_model"]
)
print(
    "Number of ML features:",
    len(ML_FEATURES)
)
print("====================================")

========== PHASE 8 INPUTS ==========
Final candidate exists: True
Evaluation file exists: True
Test feature file exists: True
Selected model: XGBoost_Baseline
Number of ML features: 6


In [2]:
deployment_model = xgb.XGBClassifier()

deployment_model.load_model(
    FINAL_MODEL_CANDIDATE_PATH
)

if deployment_model is None:
    raise ValueError(
        "Model loading failed."
    )

print("========== MODEL LOAD TEST ==========")
print(
    "Model loaded successfully:",
    True
)

print(
    "Model type:",
    "XGBoost"
)

print(
    "Model file:",
    FINAL_MODEL_CANDIDATE_PATH.name
)

print("=====================================")

========== MODEL LOAD TEST ==========
Model loaded successfully: True
Model type: XGBoost
Model file: xgboost_final_candidate.json


In [3]:
required_columns = ML_FEATURES + ["is_fraud"]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing deployment columns: {missing_columns}"
    )

deployment_feature_data = df[
    ML_FEATURES
].copy()

for column in ML_FEATURES:
    deployment_feature_data[column] = pd.to_numeric(
        deployment_feature_data[column],
        errors="coerce"
    )

if deployment_feature_data.isnull().any().any():
    raise ValueError(
        "Missing values found in deployment features."
    )

if not np.isfinite(
    deployment_feature_data.to_numpy(
        dtype=float
    )
).all():
    raise ValueError(
        "Invalid numeric values found in deployment features."
    )

print("========== FEATURE CONTRACT TEST ==========")
print(
    "Feature count:",
    len(ML_FEATURES)
)

print(
    "Feature order:",
    ML_FEATURES
)

print(
    "All required features present:",
    True
)

print(
    "All deployment values valid:",
    True
)

print("===========================================")

========== FEATURE CONTRACT TEST ==========
Feature count: 6
Feature order: ['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']
All required features present: True
All deployment values valid: True


In [4]:
DEPLOYMENT_MODEL_PATH = (
    MODELS_DIR
    / "xgboost_fraud_detector.json"
)

shutil.copy2(
    FINAL_MODEL_CANDIDATE_PATH,
    DEPLOYMENT_MODEL_PATH
)

print("========== DEPLOYMENT MODEL ==========")
print(
    "Deployment model:",
    DEPLOYMENT_MODEL_PATH
)

print(
    "Deployment model exists:",
    DEPLOYMENT_MODEL_PATH.exists()
)

print("======================================")

========== DEPLOYMENT MODEL ==========
Deployment model: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\xgboost_fraud_detector.json
Deployment model exists: True


In [5]:
FEATURE_COLUMNS_PATH = (
    MODELS_DIR
    / "feature_columns.json"
)

feature_contract = {
    "features": ML_FEATURES,
    "feature_count": len(ML_FEATURES)
}

with open(
    FEATURE_COLUMNS_PATH,
    "w"
) as file:
    json.dump(
        feature_contract,
        file,
        indent=4
    )

print("========== FEATURE CONTRACT SAVED ==========")
print(
    "Feature contract:",
    FEATURE_COLUMNS_PATH
)

print(
    "File exists:",
    FEATURE_COLUMNS_PATH.exists()
)

print("============================================")

========== FEATURE CONTRACT SAVED ==========
Feature contract: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\feature_columns.json
File exists: True


In [6]:
MODEL_INFO_PATH = (
    MODELS_DIR
    / "model_info.json"
)

model_info = {
    "model": "XGBoost",
    "model_version": "2.0",
    "selected_model": final_evaluation["selected_model"],
    "feature_count": len(ML_FEATURES),
    "features": ML_FEATURES,
    "evaluation": {
        "accuracy": final_evaluation["accuracy"],
        "precision": final_evaluation["precision"],
        "recall": final_evaluation["recall"],
        "f1": final_evaluation["f1"],
        "roc_auc": final_evaluation["roc_auc"],
        "pr_auc": final_evaluation["pr_auc"]
    }
}

with open(
    MODEL_INFO_PATH,
    "w"
) as file:
    json.dump(
        model_info,
        file,
        indent=4
    )

print("========== MODEL INFO SAVED ==========")
print(
    "Model info file:",
    MODEL_INFO_PATH
)

print(
    "File exists:",
    MODEL_INFO_PATH.exists()
)

print(
    "Model version:",
    model_info["model_version"]
)

print("=======================================")

========== MODEL INFO SAVED ==========
Model info file: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\model_info.json
File exists: True
Model version: 2.0


In [7]:
DEPLOYMENT_CONFIG_PATH = (
    MODELS_DIR
    / "deployment_config.json"
)

deployment_config = {
    "model": "XGBoost",
    "model_version": "2.0",
    "classification_threshold": 0.5,
    "feature_count": len(ML_FEATURES),
    "features": ML_FEATURES,
    "prediction_type": "fraud_probability"
}

with open(
    DEPLOYMENT_CONFIG_PATH,
    "w"
) as file:
    json.dump(
        deployment_config,
        file,
        indent=4
    )

print("========== DEPLOYMENT CONFIG ==========")
print(
    "Config file:",
    DEPLOYMENT_CONFIG_PATH
)

print(
    "File exists:",
    DEPLOYMENT_CONFIG_PATH.exists()
)

print(
    "Classification threshold:",
    deployment_config[
        "classification_threshold"
    ]
)

print("=======================================")

========== DEPLOYMENT CONFIG ==========
Config file: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\deployment_config.json
File exists: True
Classification threshold: 0.5


In [8]:
DEPLOYMENT_MANIFEST_PATH = (
    MODELS_DIR
    / "deployment_manifest.json"
)

deployment_manifest = {
    "deployment_name": "StreamSentinel_ML_V2",
    "model_file": "xgboost_fraud_detector.json",
    "feature_file": "feature_columns.json",
    "model_info_file": "model_info.json",
    "config_file": "deployment_config.json",
    "model_type": "XGBoost",
    "model_version": "2.0",
    "feature_count": len(ML_FEATURES),
    "features": ML_FEATURES
}

with open(
    DEPLOYMENT_MANIFEST_PATH,
    "w"
) as file:
    json.dump(
        deployment_manifest,
        file,
        indent=4
    )

print("========== DEPLOYMENT MANIFEST ==========")
print(
    "Manifest:",
    DEPLOYMENT_MANIFEST_PATH
)

print(
    "File exists:",
    DEPLOYMENT_MANIFEST_PATH.exists()
)

print("=========================================")

========== DEPLOYMENT MANIFEST ==========
Manifest: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models\deployment_manifest.json
File exists: True


In [9]:
FEATURE_CONTRACT_CODE = '''FEATURE_COLUMNS = [
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

FEATURE_COUNT = len(FEATURE_COLUMNS)
MODEL_VERSION = "2.0"
MODEL_NAME = "XGBoost"
'''

FEATURE_CONTRACT_PATH = (
    SRC_DIR
    / "feature_contract.py"
)

FEATURE_CONTRACT_PATH.write_text(
    FEATURE_CONTRACT_CODE,
    encoding="utf-8"
)

print("========== FEATURE CONTRACT MODULE ==========")
print(
    "File:",
    FEATURE_CONTRACT_PATH
)

print(
    "File exists:",
    FEATURE_CONTRACT_PATH.exists()
)

print("=============================================")

========== FEATURE CONTRACT MODULE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\feature_contract.py
File exists: True


In [10]:
ML_INFERENCE_CODE = '''from pathlib import Path
import json
import numpy as np
import pandas as pd
import xgboost as xgb

from feature_contract import (
    FEATURE_COLUMNS,
    FEATURE_COUNT,
    MODEL_NAME,
    MODEL_VERSION
)

SRC_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = SRC_DIR.parent
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_PATH = MODELS_DIR / "xgboost_fraud_detector.json"
CONFIG_PATH = MODELS_DIR / "deployment_config.json"
FEATURE_PATH = MODELS_DIR / "feature_columns.json"

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}"
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Deployment config not found: {CONFIG_PATH}"
    )

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Feature contract file not found: {FEATURE_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    CONFIG = json.load(file)

with open(FEATURE_PATH, "r", encoding="utf-8") as file:
    SAVED_FEATURE_CONTRACT = json.load(file)

saved_features = SAVED_FEATURE_CONTRACT["features"]

if saved_features != FEATURE_COLUMNS:
    raise ValueError(
        "Saved feature order does not match feature_contract.py"
    )

if len(saved_features) != FEATURE_COUNT:
    raise ValueError(
        "Saved feature count does not match feature_contract.py"
    )

CLASSIFICATION_THRESHOLD = float(
    CONFIG["classification_threshold"]
)

MODEL = xgb.XGBClassifier()
MODEL.load_model(MODEL_PATH)

def predict_transaction(transaction):
    if not isinstance(transaction, dict):
        raise TypeError(
            "Transaction must be a dictionary."
        )

    missing_features = [
        feature
        for feature in FEATURE_COLUMNS
        if feature not in transaction
    ]

    if missing_features:
        raise ValueError(
            f"Missing required ML features: {missing_features}"
        )

    values = {
        feature: transaction[feature]
        for feature in FEATURE_COLUMNS
    }

    input_df = pd.DataFrame(
        [values],
        columns=FEATURE_COLUMNS
    )

    for feature in FEATURE_COLUMNS:
        input_df[feature] = pd.to_numeric(
            input_df[feature],
            errors="coerce"
        )

    if input_df.isnull().any().any():
        raise ValueError(
            "Transaction contains missing or invalid feature values."
        )

    input_array = input_df.to_numpy(
        dtype=float
    )

    if not np.isfinite(input_array).all():
        raise ValueError(
            "Transaction contains non-finite feature values."
        )

    fraud_probability = float(
        MODEL.predict_proba(input_df)[0, 1]
    )

    prediction = int(
        fraud_probability >= CLASSIFICATION_THRESHOLD
    )

    prediction_label = (
        "Fraud"
        if prediction == 1
        else "Legitimate"
    )

    return {
        "ml_score": fraud_probability,
        "ml_prediction": prediction,
        "ml_label": prediction_label,
        "model": MODEL_NAME,
        "model_version": MODEL_VERSION
    }
'''

ML_INFERENCE_PATH = (
    SRC_DIR
    / "ml_inference.py"
)

ML_INFERENCE_PATH.write_text(
    ML_INFERENCE_CODE,
    encoding="utf-8"
)

print("========== ML INFERENCE MODULE ==========")
print(
    "File:",
    ML_INFERENCE_PATH
)

print(
    "File exists:",
    ML_INFERENCE_PATH.exists()
)

print("==========================================")

========== ML INFERENCE MODULE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\ml_inference.py
File exists: True


In [11]:
ML_SERVICE_CODE = '''from ml_inference import predict_transaction

def get_ml_score(transaction):
    if not isinstance(transaction, dict):
        raise TypeError(
            "Transaction must be a dictionary."
        )

    return predict_transaction(
        transaction
    )
'''

ML_SERVICE_PATH = (
    SRC_DIR
    / "ml_service.py"
)

ML_SERVICE_PATH.write_text(
    ML_SERVICE_CODE,
    encoding="utf-8"
)

print("========== ML SERVICE MODULE ==========")
print(
    "File:",
    ML_SERVICE_PATH
)

print(
    "File exists:",
    ML_SERVICE_PATH.exists()
)

print("=======================================")

========== ML SERVICE MODULE ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src\ml_service.py
File exists: True


In [12]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_service import get_ml_score

print("========== SERVICE IMPORT TEST ==========")
print(
    "ML service imported successfully:",
    callable(get_ml_score)
)

print("=========================================")

========== SERVICE IMPORT TEST ==========
ML service imported successfully: True


In [13]:
sample_transaction = (
    df.iloc[0][ML_FEATURES]
    .to_dict()
)

sample_result = get_ml_score(
    sample_transaction
)

print("========== DEPLOYMENT PREDICTION ==========")
print(
    "Transaction fields:",
    len(sample_transaction)
)

print(
    "ML result:",
    sample_result
)

print("===========================================")

========== DEPLOYMENT PREDICTION ==========
Transaction fields: 6
ML result: {'ml_score': 0.07883047312498093, 'ml_prediction': 0, 'ml_label': 'Legitimate', 'model': 'XGBoost', 'model_version': '2.0'}


In [14]:
fraud_rows = df[
    df["is_fraud"] == 1
]

legitimate_rows = df[
    df["is_fraud"] == 0
]

if fraud_rows.empty:
    raise ValueError(
        "No fraud transaction available in test features."
    )

if legitimate_rows.empty:
    raise ValueError(
        "No legitimate transaction available in test features."
    )

fraud_transaction = (
    fraud_rows.iloc[0][ML_FEATURES]
    .to_dict()
)

legitimate_transaction = (
    legitimate_rows.iloc[0][ML_FEATURES]
    .to_dict()
)

fraud_result = get_ml_score(
    fraud_transaction
)

legitimate_result = get_ml_score(
    legitimate_transaction
)

print("========== FRAUD / LEGITIMATE TEST ==========")

print()
print("Fraud transaction:")
print(fraud_result)

print()
print("Legitimate transaction:")
print(legitimate_result)

print("=============================================")

========== FRAUD / LEGITIMATE TEST ==========

Fraud transaction:
{'ml_score': 0.9653146266937256, 'ml_prediction': 1, 'ml_label': 'Fraud', 'model': 'XGBoost', 'model_version': '2.0'}

Legitimate transaction:
{'ml_score': 0.07883047312498093, 'ml_prediction': 0, 'ml_label': 'Legitimate', 'model': 'XGBoost', 'model_version': '2.0'}


In [15]:
required_response_fields = [
    "ml_score",
    "ml_prediction",
    "ml_label",
    "model",
    "model_version"
]

response_fields_valid = all(
    field in sample_result
    for field in required_response_fields
)

score_valid = (
    0 <= sample_result["ml_score"] <= 1
)

prediction_valid = (
    sample_result["ml_prediction"]
    in [0, 1]
)

label_valid = (
    sample_result["ml_label"]
    in ["Fraud", "Legitimate"]
)

model_valid = (
    sample_result["model"] == "XGBoost"
)

version_valid = (
    sample_result["model_version"] == "2.0"
)

print("========== PRODUCTION RESPONSE VALIDATION ==========")

print(
    "Required fields present:",
    response_fields_valid
)

print(
    "ML score valid:",
    score_valid
)

print(
    "Prediction valid:",
    prediction_valid
)

print(
    "Label valid:",
    label_valid
)

print(
    "Model valid:",
    model_valid
)

print(
    "Model version valid:",
    version_valid
)

print(
    "Overall response valid:",
    all([
        response_fields_valid,
        score_valid,
        prediction_valid,
        label_valid,
        model_valid,
        version_valid
    ])
)

print("====================================================")

========== PRODUCTION RESPONSE VALIDATION ==========
Required fields present: True
ML score valid: True
Prediction valid: True
Label valid: True
Model valid: True
Model version valid: True
Overall response valid: True


In [16]:
test_sample = df.head(
    min(100, len(df))
)

deployment_results = []

for _, row in test_sample.iterrows():

    transaction = (
        row[ML_FEATURES]
        .to_dict()
    )

    result = get_ml_score(
        transaction
    )

    deployment_results.append(
        result
    )

deployment_results_df = pd.DataFrame(
    deployment_results
)

all_scores_valid = (
    deployment_results_df[
        "ml_score"
    ]
    .between(0, 1)
    .all()
)

all_predictions_valid = (
    deployment_results_df[
        "ml_prediction"
    ]
    .isin([0, 1])
    .all()
)

all_labels_valid = (
    deployment_results_df[
        "ml_label"
    ]
    .isin(
        ["Fraud", "Legitimate"]
    )
    .all()
)

print("========== PRODUCTION BATCH TEST ==========")

print(
    "Transactions processed:",
    len(deployment_results_df)
)

print(
    "All scores valid:",
    all_scores_valid
)

print(
    "All predictions valid:",
    all_predictions_valid
)

print(
    "All labels valid:",
    all_labels_valid
)

print(
    "Batch test passed:",
    all([
        len(deployment_results_df)
        == len(test_sample),
        all_scores_valid,
        all_predictions_valid,
        all_labels_valid
    ])
)

print("===========================================")

========== PRODUCTION BATCH TEST ==========
Transactions processed: 100
All scores valid: True
All predictions valid: True
All labels valid: True
Batch test passed: True


In [17]:
required_deployment_files = [
    DEPLOYMENT_MODEL_PATH,
    FEATURE_COLUMNS_PATH,
    MODEL_INFO_PATH,
    DEPLOYMENT_CONFIG_PATH,
    DEPLOYMENT_MANIFEST_PATH,
    FEATURE_CONTRACT_PATH,
    ML_INFERENCE_PATH,
    ML_SERVICE_PATH
]

deployment_files_valid = all(
    path.exists()
    for path in required_deployment_files
)

print("========== DEPLOYMENT FILE CHECK ==========")

for path in required_deployment_files:
    print(
        path.name + ":",
        path.exists()
    )

print()

print(
    "All deployment files present:",
    deployment_files_valid
)

print("===========================================")

========== DEPLOYMENT FILE CHECK ==========
xgboost_fraud_detector.json: True
feature_columns.json: True
model_info.json: True
deployment_config.json: True
deployment_manifest.json: True
feature_contract.py: True
ml_inference.py: True
ml_service.py: True

All deployment files present: True


In [18]:
deployment_validation = {
    "deployment_status": "READY",
    "model": "XGBoost",
    "model_version": "2.0",
    "feature_count": len(ML_FEATURES),
    "features": ML_FEATURES,
    "classification_threshold": deployment_config[
        "classification_threshold"
    ],
    "model_loaded": True,
    "service_available": callable(
        get_ml_score
    ),
    "response_valid": all([
        response_fields_valid,
        score_valid,
        prediction_valid,
        label_valid,
        model_valid,
        version_valid
    ]),
    "batch_test_passed": all([
        len(deployment_results_df)
        == len(test_sample),
        all_scores_valid,
        all_predictions_valid,
        all_labels_valid
    ]),
    "all_deployment_files_present": deployment_files_valid
}

DEPLOYMENT_VALIDATION_PATH = (
    METRICS_DIR
    / "deployment_validation.json"
)

with open(
    DEPLOYMENT_VALIDATION_PATH,
    "w"
) as file:
    json.dump(
        deployment_validation,
        file,
        indent=4
    )

print("========== DEPLOYMENT VALIDATION SAVED ==========")
print(
    "File:",
    DEPLOYMENT_VALIDATION_PATH
)

print(
    "File exists:",
    DEPLOYMENT_VALIDATION_PATH.exists()
)

print("=================================================")

========== DEPLOYMENT VALIDATION SAVED ==========
File: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\deployment_validation.json
File exists: True


In [19]:
final_deployment_ready = all([
    DEPLOYMENT_MODEL_PATH.exists(),
    FEATURE_COLUMNS_PATH.exists(),
    MODEL_INFO_PATH.exists(),
    DEPLOYMENT_CONFIG_PATH.exists(),
    DEPLOYMENT_MANIFEST_PATH.exists(),
    FEATURE_CONTRACT_PATH.exists(),
    ML_INFERENCE_PATH.exists(),
    ML_SERVICE_PATH.exists(),
    callable(get_ml_score),
    response_fields_valid,
    score_valid,
    prediction_valid,
    label_valid,
    model_valid,
    version_valid,
    all_scores_valid,
    all_predictions_valid,
    all_labels_valid
])

print()
print("================================================")
print("     STREAMSENTINEL V2 — PHASE 8 SUMMARY")
print("================================================")

print()

print(
    "Model:",
    "XGBoost"
)

print(
    "Model version:",
    "2.0"
)

print(
    "Feature count:",
    len(ML_FEATURES)
)

print(
    "Features:",
    ML_FEATURES
)

print(
    "Classification threshold:",
    deployment_config[
        "classification_threshold"
    ]
)

print()

print(
    "Deployment model exists:",
    DEPLOYMENT_MODEL_PATH.exists()
)

print(
    "Feature contract exists:",
    FEATURE_COLUMNS_PATH.exists()
)

print(
    "Model info exists:",
    MODEL_INFO_PATH.exists()
)

print(
    "Deployment config exists:",
    DEPLOYMENT_CONFIG_PATH.exists()
)

print(
    "Deployment manifest exists:",
    DEPLOYMENT_MANIFEST_PATH.exists()
)

print(
    "Feature contract module exists:",
    FEATURE_CONTRACT_PATH.exists()
)

print(
    "ML inference module exists:",
    ML_INFERENCE_PATH.exists()
)

print(
    "ML service exists:",
    ML_SERVICE_PATH.exists()
)

print(
    "Production service available:",
    callable(get_ml_score)
)

print(
    "Production response valid:",
    all([
        response_fields_valid,
        score_valid,
        prediction_valid,
        label_valid,
        model_valid,
        version_valid
    ])
)

print(
    "100-transaction batch test:",
    all([
        len(deployment_results_df)
        == len(test_sample),
        all_scores_valid,
        all_predictions_valid,
        all_labels_valid
    ])
)

print()
print(
    "FINAL ML DEPLOYMENT STATUS:",
    "READY"
    if final_deployment_ready
    else "NOT READY"
)

print(
    "Overall verification:",
    final_deployment_ready
)

print("================================================")


     STREAMSENTINEL V2 — PHASE 8 SUMMARY

Model: XGBoost
Model version: 2.0
Feature count: 6
Features: ['amount', 'amount_vs_avg_ratio', 'txn_count_last_5min', 'time_since_last_txn_sec', 'distance_from_last_location_km', 'merchant_category_is_new_for_user']
Classification threshold: 0.5

Deployment model exists: True
Feature contract exists: True
Model info exists: True
Deployment config exists: True
Deployment manifest exists: True
Feature contract module exists: True
ML inference module exists: True
ML service exists: True
Production service available: True
Production response valid: True
100-transaction batch test: True

FINAL ML DEPLOYMENT STATUS: READY
Overall verification: True
